# Multilingual Interview Bot — Quality Testing

Tests MedGemma's ability to conduct patient intake interviews in 6 non-English languages:
- **Singapore**: Mandarin (zh), Malay (ms), Tamil (ta)
- **US**: Spanish (es), Vietnamese (vi), Arabic (ar)

Each scenario feeds scripted patient responses through the interview pipeline and scores:
1. Bot responded in the target language
2. Extracted data is in English
3. Chief complaint correctly identified
4. Medications found
5. Allergies found
6. Red flags detected
7. ESI level in expected range

**Decision gate**: If ≥ 4/6 languages score ≥ 60%, we proceed to Phase B (app integration).

In [ ]:
# Cell 1: Setup
import sys
import os
import json
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path(os.getcwd()).parent if "notebooks" in os.getcwd() else Path(os.getcwd())
sys.path.insert(0, str(PROJECT_ROOT))

from src.medgemma.interview import PatientInterviewer, InterviewSession, create_session
from src.medgemma.interview_prompts import (
    INTERVIEW_SYSTEM_PROMPT,
    get_system_prompt,
    get_opening_question,
    SUPPORTED_LANGUAGES,
)
from src.medgemma.client import get_medgemma_client

import langdetect
import pandas as pd

print(f"Project root: {PROJECT_ROOT}")
print(f"Supported languages: {list(SUPPORTED_LANGUAGES.keys())}")
print("Setup complete.")

In [ ]:
# Cell 2: Multilingual system prompt preview
# Show what the augmented system prompt looks like for each language

for code, (name, bcp47) in SUPPORTED_LANGUAGES.items():
    prompt = get_system_prompt(code)
    suffix = "(base prompt)" if code == "en" else f"(+{len(prompt) - len(INTERVIEW_SYSTEM_PROMPT)} chars)"
    print(f"{code} ({name}): {suffix}")

print("\n--- Example: Spanish system prompt ---")
print(get_system_prompt("es"))

In [ ]:
# Cell 3: Patient scenarios — scripted patient responses with ground truth

SCENARIOS = [
    {
        "id": "zh_chest_pain",
        "language": "zh",
        "language_name": "Mandarin Chinese",
        "demographics": "45M",
        "chief_complaint_en": "chest pain",
        "responses": [
            "我胸口很痛，从今天早上开始的。",  # Chest pain since this morning
            "疼痛是压迫感，大概7分。走路的时候会更严重。",  # Crushing, 7/10, worse with walking
            "没有发烧，但是有点出汗和恶心。",  # No fever, sweating and nausea
            "我有高血压和糖尿病。",  # HTN and diabetes
            "我吃降压药和二甲双胍。",  # Antihypertensive and metformin
            "我对青霉素过敏。",  # Penicillin allergy
            "是的，没错。",  # Confirmation
        ],
        "ground_truth": {
            "chief_complaint": ["chest pain", "chest", "pain"],
            "medications": ["metformin", "antihypertensive"],
            "allergies": ["penicillin"],
            "red_flags": ["chest pain", "diaphoresis", "nausea"],
            "esi_range": [1, 2],
        },
    },
    {
        "id": "ms_abdominal",
        "language": "ms",
        "language_name": "Malay",
        "demographics": "60M",
        "chief_complaint_en": "abdominal pain",
        "responses": [
            "Perut saya sakit di bahagian kanan bawah.",  # RLQ abdominal pain
            "Mula semalam. Sakit semakin teruk, 8 daripada 10.",  # Since yesterday, 8/10
            "Ada demam sikit dan rasa loya.",  # Low fever, nausea
            "Saya ada kencing manis dan darah tinggi.",  # DM and HTN
            "Saya makan ubat gula dan ubat darah tinggi.",  # DM and HTN meds
            "Tidak ada alahan ubat.",  # No drug allergies
            "Ya, betul.",  # Confirmation
        ],
        "ground_truth": {
            "chief_complaint": ["abdominal pain", "abdomen", "RLQ"],
            "medications": ["diabetes", "hypertension", "antihypertensive", "metformin", "oral hypoglycemic"],
            "allergies": ["none", "no known", "nkda"],
            "red_flags": ["fever", "RLQ", "appendicitis"],
            "esi_range": [2, 3],
        },
    },
    {
        "id": "ta_headache",
        "language": "ta",
        "language_name": "Tamil",
        "demographics": "35F pregnant",
        "chief_complaint_en": "severe headache",
        "responses": [
            "எனக்கு கடுமையான தலைவலி இருக்கிறது.",  # Severe headache
            "இரண்டு நாட்களாக இருக்கிறது. வலி 9/10. ஒளி தொந்தரவாக இருக்கிறது.",  # 2 days, 9/10, photophobia
            "நான் 28 வாரம் கர்ப்பமாக இருக்கிறேன். கால்கள் வீங்கியிருக்கின்றன.",  # 28wks pregnant, leg swelling
            "முன்பு எந்த நோயும் இல்லை.",  # No prior conditions
            "கர்ப்ப வைட்டமின்கள் மட்டும்.",  # Prenatal vitamins only
            "சல்ஃபா மருந்துகளுக்கு ஒவ்வாமை.",  # Sulfa allergy
            "ஆம், சரி.",  # Confirmation
        ],
        "ground_truth": {
            "chief_complaint": ["headache", "head"],
            "medications": ["prenatal", "vitamin"],
            "allergies": ["sulfa"],
            "red_flags": ["headache", "pregnant", "preeclampsia", "edema", "swelling", "photophobia"],
            "esi_range": [1, 2],
        },
    },
    {
        "id": "es_sob_chf",
        "language": "es",
        "language_name": "Spanish",
        "demographics": "55F",
        "chief_complaint_en": "shortness of breath with CHF history",
        "responses": [
            "No puedo respirar bien. Me falta el aire desde ayer.",  # SOB since yesterday
            "Empeoró anoche. No puedo dormir acostada, tengo que sentarme. Diría un 7.",  # Orthopnea, 7/10
            "Tengo los tobillos hinchados y he subido de peso esta semana.",  # Ankle edema, weight gain
            "Tengo insuficiencia cardíaca y presión alta.",  # CHF and HTN
            "Tomo furosemida, lisinopril y metoprolol.",  # Furosemide, lisinopril, metoprolol
            "Soy alérgica a la aspirina.",  # Aspirin allergy
            "Sí, todo correcto.",  # Confirmation
        ],
        "ground_truth": {
            "chief_complaint": ["shortness of breath", "dyspnea", "breathing"],
            "medications": ["furosemide", "lisinopril", "metoprolol"],
            "allergies": ["aspirin"],
            "red_flags": ["dyspnea", "orthopnea", "edema", "CHF", "heart failure"],
            "esi_range": [2, 3],
        },
    },
    {
        "id": "vi_dizziness",
        "language": "vi",
        "language_name": "Vietnamese",
        "demographics": "70F",
        "chief_complaint_en": "dizziness and falls",
        "responses": [
            "Tôi bị chóng mặt và té ngã sáng nay.",  # Dizzy, fell this morning
            "Bắt đầu từ hai ngày trước. Khi đứng dậy thì chóng mặt hơn. Khoảng 6 điểm.",  # 2 days, worse standing, 6/10
            "Không sốt. Có buồn nôn nhẹ.",  # No fever, mild nausea
            "Tôi bị tiểu đường và cao huyết áp. Năm ngoái bị gãy xương hông.",  # DM, HTN, hip fracture
            "Tôi uống metformin, amlodipine, và vitamin D.",  # Metformin, amlodipine, vit D
            "Tôi dị ứng với codeine.",  # Codeine allergy
            "Đúng rồi.",  # Confirmation
        ],
        "ground_truth": {
            "chief_complaint": ["dizziness", "dizzy", "fall"],
            "medications": ["metformin", "amlodipine", "vitamin D"],
            "allergies": ["codeine"],
            "red_flags": ["fall", "dizziness", "elderly", "syncope", "orthostatic"],
            "esi_range": [2, 3],
        },
    },
    {
        "id": "ar_flank_pain",
        "language": "ar",
        "language_name": "Arabic",
        "demographics": "40F",
        "chief_complaint_en": "flank pain with hematuria",
        "responses": [
            "عندي ألم شديد في الجانب الأيمن ودم في البول.",  # Severe R flank pain + hematuria
            "بدأ من ست ساعات. الألم يأتي ويروح. شدته 9 من 10.",  # 6hrs, colicky, 9/10
            "عندي غثيان وتقيأت مرتين. لا حرارة.",  # Nausea, vomited 2x, no fever
            "عندي حصى كلى سابقة ونقرس.",  # Prior kidney stones, gout
            "آخذ الوبيورينول وإيبوبروفين عند الحاجة.",  # Allopurinol, ibuprofen PRN
            "عندي حساسية من الصبغة المستخدمة في الأشعة.",  # Contrast dye allergy
            "نعم، صحيح.",  # Confirmation
        ],
        "ground_truth": {
            "chief_complaint": ["flank pain", "kidney", "renal", "hematuria"],
            "medications": ["allopurinol", "ibuprofen"],
            "allergies": ["contrast", "dye"],
            "red_flags": ["hematuria", "severe pain", "renal colic"],
            "esi_range": [2, 3],
        },
    },
]

print(f"Loaded {len(SCENARIOS)} scenarios:")
for s in SCENARIOS:
    print(f"  {s['id']}: {s['language_name']} — {s['chief_complaint_en']} ({s['demographics']})")

In [ ]:
# Cell 4: Interview runner
# Runs each scenario through the interview pipeline

import asyncio

async def run_scenario(scenario: dict) -> dict:
    """Run a single multilingual interview scenario."""
    lang = scenario["language"]
    print(f"\n{'='*60}")
    print(f"Running: {scenario['id']} ({scenario['language_name']})")
    print(f"{'='*60}")

    # Create session with language
    session = create_session(language=lang)

    interviewer = PatientInterviewer()

    # Start interview
    start_result = await interviewer.start_interview(session)
    print(f"  Nurse: {start_result['question'][:80]}...")

    # Feed scripted responses
    results = []
    for i, response in enumerate(scenario["responses"]):
        if session.phase in ("review_and_triage", "complete"):
            break
        result = await interviewer.process_response(session, response)
        results.append(result)
        print(f"  Patient: {response[:60]}...")
        print(f"  Nurse [{result['phase']}]: {result['question'][:80]}...")

    # Generate triage
    triage = await interviewer.generate_triage(session)
    print(f"  Triage: ESI {triage.get('esi_level', '?')} — {triage.get('chief_complaint', '?')[:60]}")

    return {
        "scenario": scenario,
        "conversation": session.conversation_history,
        "extracted_data": session.extracted_data,
        "red_flags": session.red_flags,
        "triage": triage,
        "session_id": session.session_id,
    }

print("Interview runner defined.")

In [ ]:
# Cell 5: Scoring functions

from langdetect import detect, LangDetectException

def score_language_compliance(conversation: list[dict], target_lang: str) -> float:
    """Check that bot responses are in the target language (0-1)."""
    bot_messages = [m["content"] for m in conversation if m["role"] == "assistant"]
    if not bot_messages:
        return 0.0
    correct = 0
    for msg in bot_messages:
        try:
            detected = detect(msg)
            # langdetect uses 'zh-cn'/'zh-tw' for Chinese
            detected_base = detected.split("-")[0]
            if detected_base == target_lang:
                correct += 1
        except LangDetectException:
            pass
    return correct / len(bot_messages)


def score_extracted_data_english(extracted_data: dict) -> float:
    """Check that extracted data values are in English (0-1)."""
    values = []
    def collect_values(obj):
        if isinstance(obj, dict):
            for v in obj.values():
                collect_values(v)
        elif isinstance(obj, list):
            for item in obj:
                collect_values(item)
        elif isinstance(obj, str) and len(obj) > 3:
            values.append(obj)
    collect_values(extracted_data)
    if not values:
        return 0.5  # No data to check
    english_count = 0
    for v in values:
        try:
            if detect(v) == "en":
                english_count += 1
        except LangDetectException:
            english_count += 1  # Short strings, assume OK
    return english_count / len(values)


def score_chief_complaint(triage: dict, ground_truth: dict) -> float:
    """Check if chief complaint was correctly identified (0 or 1)."""
    cc = (triage.get("chief_complaint", "") or "").lower()
    for keyword in ground_truth["chief_complaint"]:
        if keyword.lower() in cc:
            return 1.0
    return 0.0


def score_medications(triage: dict, ground_truth: dict) -> float:
    """Check if medications were captured (partial credit)."""
    found_meds = " ".join(str(m).lower() for m in (triage.get("medications", []) or []))
    # Also check extracted data
    gt_meds = ground_truth["medications"]
    if not gt_meds:
        return 1.0
    found = sum(1 for m in gt_meds if m.lower() in found_meds)
    return min(found / max(len(gt_meds), 1), 1.0)


def score_allergies(triage: dict, ground_truth: dict) -> float:
    """Check if allergies were captured (0 or 1)."""
    found_allergies = " ".join(str(a).lower() for a in (triage.get("allergies", []) or []))
    gt_allergies = ground_truth["allergies"]
    for a in gt_allergies:
        if a.lower() in found_allergies:
            return 1.0
    return 0.0


def score_red_flags(triage: dict, red_flags: list, ground_truth: dict) -> float:
    """Check if red flags were detected (partial credit)."""
    all_flags = " ".join(str(f).lower() for f in (triage.get("red_flags", []) or []) + red_flags)
    gt_flags = ground_truth["red_flags"]
    if not gt_flags:
        return 1.0
    found = sum(1 for f in gt_flags if f.lower() in all_flags)
    return min(found / max(len(gt_flags), 1), 1.0)


def score_esi(triage: dict, ground_truth: dict) -> float:
    """Check if ESI level is in expected range (0 or 1)."""
    esi = triage.get("esi_level")
    if esi is None:
        return 0.0
    return 1.0 if esi in ground_truth["esi_range"] else 0.0


def score_scenario(result: dict) -> dict:
    """Score a complete scenario run. Returns dict of individual + composite scores."""
    gt = result["scenario"]["ground_truth"]
    lang = result["scenario"]["language"]

    scores = {
        "language_compliance": score_language_compliance(result["conversation"], lang),
        "extracted_data_english": score_extracted_data_english(result["extracted_data"]),
        "chief_complaint": score_chief_complaint(result["triage"], gt),
        "medications": score_medications(result["triage"], gt),
        "allergies": score_allergies(result["triage"], gt),
        "red_flags": score_red_flags(result["triage"], result["red_flags"], gt),
        "esi_level": score_esi(result["triage"], gt),
    }

    # Weighted composite: language compliance most important (25%), rest equal
    weights = {
        "language_compliance": 0.25,
        "extracted_data_english": 0.15,
        "chief_complaint": 0.15,
        "medications": 0.10,
        "allergies": 0.10,
        "red_flags": 0.10,
        "esi_level": 0.15,
    }
    scores["composite"] = sum(scores[k] * weights[k] for k in weights)
    scores["composite_pct"] = round(scores["composite"] * 100, 1)

    return scores

print("Scoring functions defined.")

In [ ]:
# Cell 6: Run all scenarios

all_results = []
all_scores = []

for scenario in SCENARIOS:
    try:
        result = await run_scenario(scenario)
        scores = score_scenario(result)
        all_results.append(result)
        all_scores.append({
            "id": scenario["id"],
            "language": scenario["language_name"],
            **scores,
        })
        print(f"  => Score: {scores['composite_pct']}%")
    except Exception as e:
        print(f"  => ERROR: {e}")
        all_scores.append({
            "id": scenario["id"],
            "language": scenario["language_name"],
            "composite_pct": 0,
            "error": str(e),
        })

print(f"\nCompleted {len(all_results)}/{len(SCENARIOS)} scenarios.")

In [ ]:
# Cell 7: Report table

df = pd.DataFrame(all_scores)

# Format for display
display_cols = ["id", "language", "language_compliance", "extracted_data_english",
                "chief_complaint", "medications", "allergies", "red_flags",
                "esi_level", "composite_pct"]
display_df = df[[c for c in display_cols if c in df.columns]].copy()

# Color-code pass/fail
def highlight_score(val):
    if isinstance(val, (int, float)):
        if val >= 0.6 or (isinstance(val, float) and val > 1 and val >= 60):
            return "background-color: #d4edda"  # Green
        elif val >= 0.4 or (isinstance(val, float) and val > 1 and val >= 40):
            return "background-color: #fff3cd"  # Yellow
        else:
            return "background-color: #f8d7da"  # Red
    return ""

score_cols = [c for c in display_df.columns if c not in ("id", "language")]
styled = display_df.style.map(highlight_score, subset=score_cols)

# Summary stats
passing = sum(1 for s in all_scores if s.get("composite_pct", 0) >= 60)
avg_score = sum(s.get("composite_pct", 0) for s in all_scores) / max(len(all_scores), 1)

print(f"\n{'='*60}")
print(f"RESULTS: {passing}/6 passing (>= 60%), average {avg_score:.1f}%")
print(f"{'='*60}\n")

styled

In [ ]:
# Cell 8: Save conversation logs

LOG_DIR = PROJECT_ROOT / "data" / "multilingual_interview_logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

for result, scores in zip(all_results, all_scores):
    scenario = result["scenario"]
    log = {
        "scenario_id": scenario["id"],
        "language": scenario["language"],
        "language_name": scenario["language_name"],
        "demographics": scenario["demographics"],
        "conversation": result["conversation"],
        "extracted_data": result["extracted_data"],
        "triage": result["triage"],
        "red_flags": result["red_flags"],
        "scores": {k: v for k, v in scores.items() if k not in ("id", "language")},
        "session_id": result["session_id"],
    }
    filename = f"{scenario['language']}_{scenario['id'].split('_', 1)[1]}.json"
    filepath = LOG_DIR / filename
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(log, f, indent=2, ensure_ascii=False)
    print(f"Saved: {filepath.name}")

print(f"\nAll logs saved to {LOG_DIR}")

In [ ]:
# Cell 9: Decision gate

PASS_THRESHOLD = 60  # percent
MIN_PASSING = 4  # out of 6 languages

passing_langs = [s for s in all_scores if s.get("composite_pct", 0) >= PASS_THRESHOLD]
num_passing = len(passing_langs)

print(f"\n{'='*60}")
print(f"DECISION GATE")
print(f"{'='*60}")
print(f"Threshold: >= {PASS_THRESHOLD}% composite score")
print(f"Required: >= {MIN_PASSING}/{len(SCENARIOS)} languages passing")
print(f"Result: {num_passing}/{len(SCENARIOS)} passing")
print()

for s in all_scores:
    status = "PASS" if s.get("composite_pct", 0) >= PASS_THRESHOLD else "FAIL"
    print(f"  [{status}] {s['language']}: {s.get('composite_pct', 0):.1f}%")

print()
if num_passing >= MIN_PASSING:
    print("PROCEED TO PHASE B: Multilingual support quality is sufficient.")
else:
    print("DO NOT PROCEED: Multilingual quality below threshold. Review prompts and retry.")